# Gold `dim_province`

One row per reporting unit (100): a province, an HUC that DOT reports separately, the City of Isabela, or NCR.

| | |
| --- | --- |
| **Key** | `province_psgc`, the 10-digit PSGC code stored as text |
| **Built from** | `03_silver.geo_reporting_unit`, plus `city_class` from `geo_psgc` |
| **Used by** | `fact_province_year` (foreign key) and the Gold views |

`city_class` is not on the unit table, so it is joined from `geo_psgc` on the PSGC code. It is NULL for provinces and NCR, and filled for HUCs and the independent city.

**Run log:** the run is `STARTED` first and is marked `SUCCESS` or `SUCCESS_WITH_REVIEW` only after the build, checks, write **and** column comments all finish. Any error marks this `run_id` as `FAILED`, records which tables were already written, saves the error text, and re-raises the original error.

## Configuration, helpers and run log

`dq(...)` records each check. `update_run_log(...)` and `save_dq_results()` are separate helpers so the error handler in Step 4 can use them. The row and table counts in the run log come from what was actually written.

In [0]:
from pyspark.sql import functions as F
from functools import reduce
import uuid
from datetime import datetime

CATALOG = "tuklas_dev"
SILVER_SCHEMA = "03_silver"
GOLD_SCHEMA = "04_gold"
DQ_SCHEMA = "06_data_quality"
NOTEBOOK_NAME = "01_gold_dim_province"
run_id = f"gold_dim_province_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

TBL_DIM_PROVINCE = "dim_province"
EXPECTED_UNITS = 100
UNITS_WITHOUT_CITY_CLASS = ("Province", "NCR")     # unit_type values from 03_silver.geo_reporting_unit; every other unit is a city

def silver_table(n): return f"`{CATALOG}`.`{SILVER_SCHEMA}`.`{n}`"
def gold_table(n):   return f"`{CATALOG}`.`{GOLD_SCHEMA}`.`{n}`"
def dq_table(n):     return f"`{CATALOG}`.`{DQ_SCHEMA}`.`{n}`"

# ── Run state ────────────────────────────────────────────────────
dq_results = []
written = {}          # Gold tables fully written in this run: name -> rows
dq_saved = False

def dq(check_id, table_name, severity, failed_rows, expected, observed, sample_keys=None, notes=""):
    """Record one data-quality check result. failed_rows = 0 means PASS."""
    status = "PASS" if failed_rows == 0 else "FAIL"
    keys = " | ".join(str(k) for k in list(sample_keys)[:20]) if sample_keys else None
    dq_results.append((run_id, NOTEBOOK_NAME, check_id, table_name, severity, status, int(failed_rows),
                       str(expected), str(observed), keys, notes))
    marker = "✓" if status == "PASS" else ("✗" if severity == "BLOCK" else "!")
    print(f"{marker} {check_id:<11} {severity:<6} {status:<4} | {table_name} | expected {expected} | observed {observed}"
          + (f" | e.g. {keys}" if keys else ""))

def sample(df, *cols, n=20):
    return [" / ".join("" if r[c] is None else str(r[c]) for c in cols) for r in df.select(*cols).limit(n).collect()]

def dq_counts():
    blocks = [r for r in dq_results if r[4] == "BLOCK" and r[5] == "FAIL"]
    reviews = [r for r in dq_results if r[4] == "REVIEW" and r[5] == "FAIL"]
    return blocks, reviews

def update_run_log(status, notes):
    """Update this run_id's row. Table and row counts come from what was actually written so far."""
    blocks, reviews = dq_counts()
    spark.sql(f"""
    UPDATE {gold_table('gold_run_log')}
    SET status = :status, completed_at = current_timestamp(), tables_written = :tw, rows_written = :rw,
        dq_checks_run = :dc, dq_block_failures = :db, dq_review_flags = :dr, notes = :notes
    WHERE run_id = :run_id
    """, args={"status": status, "tw": len(written), "rw": sum(written.values()), "dc": len(dq_results),
               "db": len(blocks), "dr": len(reviews), "notes": notes[:2000], "run_id": run_id})

def save_dq_results():
    """Append this run's check results to dq_results once, even when the run stops early."""
    global dq_saved
    if dq_saved or not dq_results:
        return
    dq_df = (spark.createDataFrame(dq_results,
        "run_id STRING, notebook_name STRING, check_id STRING, table_name STRING, severity STRING, status STRING, "
        "failed_rows BIGINT, expected STRING, observed STRING, sample_keys STRING, notes STRING")
        .withColumn("checked_at", F.current_timestamp()))
    display(dq_df.select("check_id", "severity", "status", "failed_rows", "expected", "observed", "sample_keys"))
    dq_df.select("run_id", "notebook_name", "check_id", "table_name", "severity", "status", "failed_rows",
                 "expected", "observed", "sample_keys", "checked_at", "notes") \
         .write.format("delta").mode("append").saveAsTable(dq_table("dq_results"))
    dq_saved = True

spark.sql(f"""
INSERT INTO {gold_table('gold_run_log')}
(run_id, notebook_name, started_at, completed_at, tables_written, rows_written,
 dq_checks_run, dq_block_failures, dq_review_flags, status, notes)
VALUES (:run_id, :nb, current_timestamp(), NULL, 0, 0, 0, 0, 0, 'STARTED', 'Notebook execution started')
""", args={"run_id": run_id, "nb": NOTEBOOK_NAME})
print(f"Run ID: {run_id}")

## Step 1 — Build

One row per reporting unit. `province_psgc` is the unit's PSGC code. `city_class` is joined from `geo_psgc` on the same code with a left join, so no unit is dropped. The join is checked in Step 2, because duplicate codes in `geo_psgc` would add rows and a city with no match would get a NULL `city_class` without warning.

In [0]:
def build_dim_province():
    unit = spark.table(silver_table("geo_reporting_unit"))
    psgc = spark.table(silver_table("geo_psgc")).select(F.col("psgc_code").alias("unit_psgc_code"), "city_class")

    dim_province = (unit.join(psgc, "unit_psgc_code", "left")
        .select(F.col("unit_psgc_code").alias("province_psgc"),
                F.col("unit_name").alias("province_name"),
                "unit_type",
                F.col("region_code").alias("region_psgc"),
                "region_name",
                "city_class",
                F.col("dot_comparability_note").alias("comparability_note"),
                F.col("psgc_version").alias("psgc_snapshot"),
                "census_population_ref",
                "census_population_ref_year"))
    display(dim_province.groupBy("unit_type").count().orderBy("unit_type"))
    return dim_province, unit

## Step 2: Data-quality checks

| ID | Check | Severity |
| --- | --- | --- |
| DQ-GLD-01 | 100 units; `province_psgc` unique, 10 digits and never NULL; failing codes are listed in `sample_keys` | BLOCK |
| DQ-GLD-01b | Name, unit type, region code and region name are filled (not NULL, not blank) for every unit | BLOCK |
| DQ-GLD-01c | No replacement characters (`�`) in names | BLOCK |
| DQ-GLD-01d | Exactly one PSGC snapshot, and it is not NULL | BLOCK |
| DQ-GLD-01e | The `geo_psgc` join keeps the unit row count (no duplicates added) | BLOCK |
| DQ-GLD-01f | `city_class` is filled for every unit that is not a province or NCR | BLOCK |

In [0]:
def run_checks(dim_province, unit):
    # DQ-GLD-01  count, unique, 10 digits, never NULL
    n = dim_province.count()
    dup_df = dim_province.groupBy("province_psgc").count().filter("count > 1")
    dup = dup_df.count()
    bad_code_df = dim_province.filter(F.col("province_psgc").isNull() | ~F.col("province_psgc").rlike(r"^[0-9]{10}$"))
    bad_code = bad_code_df.count()
    dq("DQ-GLD-01", TBL_DIM_PROVINCE, "BLOCK", dup + bad_code + (0 if n == EXPECTED_UNITS else 1),
       f"{EXPECTED_UNITS} unique 10-digit codes", f"{n} rows, {dup} duplicates, {bad_code} bad codes",
       sample(dup_df, "province_psgc") + sample(bad_code_df, "province_psgc"))

    # DQ-GLD-01b  required text columns are filled, not NULL and not blank
    def blank(c):
        return F.col(c).isNull() | (F.trim(F.col(c)) == "")
    no_name = dim_province.filter(blank("province_name") | blank("unit_type") | blank("region_psgc") | blank("region_name"))
    no_name_n = no_name.count()
    dq("DQ-GLD-01b", TBL_DIM_PROVINCE, "BLOCK", no_name_n,
       "name, unit_type and region filled for every unit", f"{no_name_n} incomplete",
       sample(no_name, "province_psgc"))

    # DQ-GLD-01c  no replacement characters
    bad_chars = dim_province.filter(F.col("province_name").contains("�"))
    bad_chars_n = bad_chars.count()
    dq("DQ-GLD-01c", TBL_DIM_PROVINCE, "BLOCK", bad_chars_n, "0 names with �", f"{bad_chars_n}",
       sample(bad_chars, "province_psgc", "province_name"))

    # DQ-GLD-01d  one snapshot, and not NULL
    snap = [r[0] for r in dim_province.select("psgc_snapshot").distinct().collect()]
    bad_snap = 0 if (len(snap) == 1 and snap[0] is not None) else 1
    dq("DQ-GLD-01d", TBL_DIM_PROVINCE, "BLOCK", bad_snap, "one non-null PSGC snapshot", f"{snap}")

    # DQ-GLD-01e  the city_class join must not add or drop unit rows
    n_unit = unit.count()
    dq("DQ-GLD-01e", TBL_DIM_PROVINCE, "BLOCK", abs(n - n_unit),
       "join keeps the unit row count", f"{n_unit} units -> {n} rows")

    # DQ-GLD-01f  every city has a city_class (provinces and NCR are not cities)
    missing_class = dim_province.filter(~F.col("unit_type").isin(*UNITS_WITHOUT_CITY_CLASS) & F.col("city_class").isNull())
    missing_class_n = missing_class.count()
    dq("DQ-GLD-01f", TBL_DIM_PROVINCE, "BLOCK", missing_class_n,
       "city_class filled for every city", f"{missing_class_n} missing",
       sample(missing_class, "province_psgc", "province_name"))

## Step 3: Write and add column comments

The comments record what each key column means. The run is not marked successful until they are done.

In [0]:
def sql_str(text):
    """Escape single quotes so a comment can sit inside a SQL string literal."""
    return text.replace("'", "''")

def write_dim_province(dim_province):
    (dim_province.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(gold_table(TBL_DIM_PROVINCE)))
    written[TBL_DIM_PROVINCE] = spark.table(gold_table(TBL_DIM_PROVINCE)).count()
    print(f"✓ {gold_table(TBL_DIM_PROVINCE)}: {written[TBL_DIM_PROVINCE]} rows")

def add_column_comments():
    DP = gold_table(TBL_DIM_PROVINCE)
    spark.sql(f"COMMENT ON TABLE {DP} IS '{sql_str('One row per reporting unit (100): province, HUC, independent city or NCR.')}'")
    comments = {
        "province_psgc": "10-digit PSGC code stored as text. Primary key; one PSGC snapshot (psgc_snapshot).",
        "province_name": "Name of the reporting unit as used across the project",
        "unit_type": "Province | HUC | Independent city | NCR",
        "region_psgc": "PSGC code of the region the unit belongs to",
        "region_name": "Name of the region",
        "city_class": "PSGC city class for HUCs and the independent city; NULL for provinces and NCR",
        "comparability_note": "Why DOT counts this unit differently from 2019 in some years; NULL when it does not",
        "psgc_snapshot": "PSGC release used for codes and names, e.g. 2Q 2026",
        "census_population_ref": "PSGC 2024 census count for the unit's area. Reference only; not an annual denominator",
        "census_population_ref_year": "Census year of census_population_ref",
    }
    for col, text in comments.items():
        spark.sql(f"ALTER TABLE {DP} ALTER COLUMN `{col}` COMMENT '{sql_str(text)}'")
    print("✓ Column comments added.")

## Step 4: Run

STARTED → build → checks → write → column comments → SUCCESS. Any error marks this run `FAILED`, records which tables were already written, saves the error text, and re-raises the original error.

In [0]:
try:
    dim_province, unit = build_dim_province()
    run_checks(dim_province, unit)
    save_dq_results()

    blocks, _ = dq_counts()
    if blocks:
        raise AssertionError("BLOCK checks failed: " + ", ".join(r[2] for r in blocks)
                             + f". See {dq_table('dq_results')} for run_id = '{run_id}'.")

    write_dim_province(dim_province)
    add_column_comments()

    _, reviews = dq_counts()
    final_status = "SUCCESS_WITH_REVIEW" if reviews else "SUCCESS"
    notes = (f"{', '.join(f'{k}={v}' for k, v in written.items())}; {len(dq_results)} checks, {len(reviews)} for review"
             + (f" ({', '.join(r[2] for r in reviews)})" if reviews else "") + ".")
    update_run_log(final_status, notes)
    print(f"Run {run_id}: {final_status}\n{notes}")

except Exception as e:
    after = (f"Tables already written: {', '.join(f'{k}={v} rows' for k, v in written.items())}."
             if written else "No Gold tables written.")
    try:
        save_dq_results()                                   # keep any checks that already ran
        update_run_log("FAILED", f"{type(e).__name__}: {e} {after}")
    except Exception as log_err:                            # never hide the original error
        print(f"Could not update the run log: {log_err}")
    raise